# Re-run: generality across models, reasoning settings and item types (Paper VI, RQ5)

This notebook recomputes the two RQ5 tables (the reasoning contrasts and the label effect on other item
types and scales) and the Fable 5.1 figures from the per-case model outputs published with release
`ai-grader-reliability-2026`, and nothing else: no model call, no corpus, no repository code. It needs
Python 3.9+ with numpy, and `per-case-model-outputs.jsonl` (or the `.gz` the report offers) in the same
folder as this notebook, or in a folder named by `RELEASE_DIR`.

Throughout, the label effect on one answer is its mark under the high attainment label minus its mark
under the low label. In the swapped-label runs that is `trueLabel − swappedLabel` in the high tail and
`swappedLabel − trueLabel` in the low tail; the label-stability runs record `high` and `low` directly.
**L** is its mean, in marks; **s** is L divided by the top of the scale the answer was marked on.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "ai-grader-reliability-2026"   # filled in by board/render_report.py
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-model-outputs.jsonl


In [2]:
from collections import defaultdict
import numpy as np

cases = defaultdict(list)
for r in per_case():
    cases[r["run"]].append(r)

def swap_rows(run):
    """Answers graded in all three swapped-label arms (control, true label, swapped label)."""
    return [r for r in cases[run] if None not in (r["control"], r["trueLabel"], r["swappedLabel"])]

def hl(rows):
    return np.array([(r["trueLabel"] - r["swappedLabel"]) if r["tail"] == "high" else
                     (r["swappedLabel"] - r["trueLabel"]) for r in rows], float)

## Reasoning settings

Each contrast holds the model, draw, prompt and arms fixed and changes only the reasoning setting. It
is computed on the answers complete in both runs: **dL** = L(more reasoning) − L(less reasoning).
Runs: GPT-5-mini at minimal (`breadth/swap-g5m-min`), default (`grader-vi/swap-gpt5mini`) and high
(`breadth/swap-g5m-high`) reasoning; Gemini 3.5 Flash Lite at default/minimal thinking (`grader-vi/swap-gemini`,
re-run `breadth/swap-lite-rerun`) and high thinking (`breadth/swap-lite-high`).

In [3]:
CONTRASTS = [  # (less, more, name), in the order the study ran them (it sets the bootstrap stream)
    ("breadth/swap-g5m-min", "breadth/swap-g5m-high", "GPT-5-mini, minimal → high"),
    ("breadth/swap-g5m-min", "grader-vi/swap-gpt5mini", "GPT-5-mini, minimal → default"),
    ("grader-vi/swap-gpt5mini", "breadth/swap-g5m-high", "GPT-5-mini, default → high"),
    ("breadth/swap-lite-rerun", "breadth/swap-lite-high", "Gemini 3.5 Flash Lite, minimal → high"),
    ("grader-vi/swap-gemini", "breadth/swap-lite-rerun", "Null: Gemini default, first run → re-run"),
]

def paired(less, more):
    b = {r["id"]: r for r in swap_rows(more)}
    a = [r for r in swap_rows(less) if r["id"] in b]
    return a, [b[r["id"]] for r in a]

point = {}
for less, more, name in CONTRASTS:
    a, b = paired(less, more)
    La, Lb = hl(a).mean(), hl(b).mean()
    point[name] = (Lb - La, La, Lb, len(a))
    print(f"{name:42s} dL {Lb - La:+.2f}   L {La:+.2f} → {Lb:+.2f}   (n = {len(a)})")

GPT-5-mini, minimal → high                 dL -0.51   L +1.05 → +0.54   (n = 220)
GPT-5-mini, minimal → default              dL -0.51   L +1.04 → +0.53   (n = 222)
GPT-5-mini, default → high                 dL -0.01   L +0.53 → +0.52   (n = 294)
Gemini 3.5 Flash Lite, minimal → high      dL +0.05   L +0.36 → +0.41   (n = 300)
Null: Gemini default, first run → re-run   dL -0.01   L +0.37 → +0.36   (n = 300)


In [4]:
# The paper's printed values (reasoning table). Compared as printed text.
def same(x, printed, nd=2):
    return f"{x:.{nd}f}" == f"{printed:.{nd}f}"

PRINTED = {"GPT-5-mini, minimal → high": (-0.51, 1.05, 0.54),
           "GPT-5-mini, minimal → default": (-0.51, 1.04, 0.53),
           "GPT-5-mini, default → high": (-0.01, 0.53, 0.52),
           "Gemini 3.5 Flash Lite, minimal → high": (0.05, 0.36, 0.41),
           "Null: Gemini default, first run → re-run": (-0.01, 0.37, 0.36)}
for name, want in PRINTED.items():
    got = point[name][:3]
    assert all(same(g, w) for g, w in zip(got, want)), (name, got, want)
assert point["GPT-5-mini, minimal → high"][3] == 220

# The text: GPT-5-mini at minimal reasoning on every answer complete in its run, the strong-tail cost,
# the failure count and the complete answers per arm.
mn = swap_rows("breadth/swap-g5m-min")
d_min = hl(mn)
hi_min = np.array([r["tail"] == "high" for r in mn])
assert same(d_min.mean(), 1.04) and same(d_min[hi_min].mean(), 1.16), (d_min.mean(), d_min[hi_min].mean())
failed = sum(r[a] is None for r in cases["breadth/swap-g5m-min"] for a in ("control", "trueLabel", "swappedLabel"))
per_arm = [sum(r[a] is not None for r in cases["breadth/swap-g5m-min"]) for a in ("control", "trueLabel", "swappedLabel")]
assert failed == 92 and len(cases["breadth/swap-g5m-min"]) * 3 == 900, failed
print(f"GPT-5-mini minimal: L {d_min.mean():+.2f} on {len(mn)} answers; strong tail {d_min[hi_min].mean():+.2f}; "
      f"{failed} of 900 gradings failed; marks returned per arm {per_arm}")
sd_teacher = np.std([r["human"] for r in cases["breadth/swap-g5m-min"]])
assert same(sd_teacher, 2.78) and same(d_min.mean() / sd_teacher, 0.38), d_min.mean() / sd_teacher
# 0.38 SD is the unrounded L over the unrounded SD. (Until 2026-10-06 the paper printed 0.37, the rounded
# 1.04 over the rounded 2.78: corrected.)
print(f"teacher SD on the 300-answer draw {sd_teacher:.4f}; L / SD = {d_min.mean():.4f} / {sd_teacher:.4f} "
      f"= {d_min.mean() / sd_teacher:.4f} (paper: 0.38)")
print("All 15 table values and the minimal-reasoning figures match the paper.")

GPT-5-mini minimal: L +1.04 on 223 answers; strong tail +1.16; 92 of 900 gradings failed; marks returned per arm [270, 267, 271]
teacher SD on the 300-answer draw 2.7842; L / SD = 1.0444 / 2.7842 = 0.3751 (paper: 0.38)
All 15 table values and the minimal-reasoning figures match the paper.


**Reasoning tokens.** Whether a setting reached the model is read from the billed reasoning tokens
each run's header records (`header.run.tokens`; the default-setting runs predate that record, and the
paper's "about 813" for GPT-5-mini's default comes from a separate ten-call probe).

In [5]:
headers = {}
import gzip, json
path = find_release_file()
with (gzip.open if path.suffix == ".gz" else open)(path, "rt", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r.get("record") != "run":
            break
        headers[r["run"]] = r["header"]

def reasoning_per_call(run):
    t = (headers[run].get("run") or {}).get("tokens") or {}
    return t.get("reasoning", 0) / max(1, t.get("withUsage", 0))

tok = {run: reasoning_per_call(run) for run in ("breadth/swap-g5m-min", "breadth/swap-g5m-high",
                                                "breadth/swap-lite-rerun", "breadth/swap-lite-high")}
print({k: round(v) for k, v in tok.items()})
assert [round(v) for v in tok.values()] == [0, 4424, 0, 1591]

{'breadth/swap-g5m-min': 0, 'breadth/swap-g5m-high': 4424, 'breadth/swap-lite-rerun': 0, 'breadth/swap-lite-high': 1591}


**Intervals.** The study resamples students within tail, the same students for both runs of a contrast
(paired), 4,000 times, percentile 95% intervals, the five contrasts in the order above from one
generator seeded 23 (`grader-model-breadth/breadth.py`, `part2`, using the attainment study's
`cluster_index`). The cell below uses the same scheme, seed and order, so on the same numpy version its
endpoints are the study's; they are compared with the printed ones, not asserted, because another numpy
version can draw a different random stream.

In [6]:
def cluster_index(groups, strata, rng):
    out = []
    for st in np.unique(strata):
        ids = np.unique(groups[strata == st])
        for g in rng.choice(ids, size=len(ids), replace=True):
            out.append(np.flatnonzero((groups == g) & (strata == st)))
    return np.concatenate(out)

PRINTED_CI = {"GPT-5-mini, minimal → high": (-0.62, -0.39), "GPT-5-mini, minimal → default": (-0.65, -0.37),
              "GPT-5-mini, default → high": (-0.10, 0.10), "Gemini 3.5 Flash Lite, minimal → high": (-0.06, 0.14),
              "Null: Gemini default, first run → re-run": (-0.07, 0.05)}
rng = np.random.default_rng(23)
for less, more, name in CONTRASTS:
    a, b = paired(less, more)
    learner = np.array([str(r["learnerId"]) for r in a])
    tail = np.array([r["tail"] == "high" for r in a])
    da, db = hl(a), hl(b)
    draws = [(db[i] - da[i]).mean() for i in (cluster_index(learner, tail, rng) for _ in range(4000))]
    lo, up = np.percentile(draws, [2.5, 97.5])
    print(f"{name:42s} dL {point[name][0]:+.2f} [{lo:+.2f}, {up:+.2f}]   paper [{PRINTED_CI[name][0]:+.2f}, {PRINTED_CI[name][1]:+.2f}]")

# The minimal-reasoning L on its own answers (the paper's +1.04 [+0.90, +1.18]) comes from the model
# table's run of all ten dumps on one stream; here it has its own stream, so endpoints can differ slightly.
learner = np.array([str(r["learnerId"]) for r in mn])
rng2 = np.random.default_rng(20261006)
draws = [d_min[cluster_index(learner, hi_min, rng2)].mean() for _ in range(4000)]
print(f"GPT-5-mini minimal, L {d_min.mean():+.2f} [{np.percentile(draws, 2.5):+.2f}, {np.percentile(draws, 97.5):+.2f}]"
      f"   paper [+0.90, +1.18]")

GPT-5-mini, minimal → high                 dL -0.51 [-0.62, -0.39]   paper [-0.62, -0.39]
GPT-5-mini, minimal → default              dL -0.51 [-0.65, -0.37]   paper [-0.65, -0.37]
GPT-5-mini, default → high                 dL -0.01 [-0.10, +0.10]   paper [-0.10, +0.10]
Gemini 3.5 Flash Lite, minimal → high      dL +0.05 [-0.06, +0.14]   paper [-0.06, +0.14]
Null: Gemini default, first run → re-run   dL -0.01 [-0.07, +0.05]   paper [-0.07, +0.05]
GPT-5-mini minimal, L +1.04 [+0.90, +1.18]   paper [+0.90, +1.18]


## Item types and scales: s

The label-stability runs apply the same high and low label pairs, rescaled to each scale, to the same
JorGPT answers marked out of 5 (`grader-label-stability/jorgpt5-*`), to Mohler short answers out of 5
(`mohler-*`) and to Menagerie Java programs on the 14-level rubric, 0–13 (`menagerie-*`). The JorGPT
0–10 column is the swapped-label runs above. s = mean(high − low) / top of the scale, over the answers
graded under both labels.

In [7]:
MODELS = {"Gemini 3.5 Flash Lite": ("grader-vi/swap-gemini", "gemini"),
          "GPT-4.1-mini": ("grader-vi/swap-gpt41mini", "gpt41mini"),
          "Qwen3-235B": ("grader-vi/swap-qwen", "qwen")}
CELLS = ["jorgpt10", "jorgpt5", "mohler", "menagerie"]

def cell(model, c):
    """(per-answer high − low in marks, scale top, cluster ids) for one model and cell."""
    swap, tag = MODELS[model]
    if c == "jorgpt10":
        rows = [r for r in cases[swap] if None not in (r["trueLabel"], r["swappedLabel"])]
        return hl(rows), 10, np.array([f'{r["tail"]}|{r["learnerId"]}' for r in rows])
    rows = [r for r in cases[f"grader-label-stability/{c}-{tag}"] if None not in (r["high"], r["low"])]
    tops = {r["max"] for r in rows}
    assert len(tops) == 1
    return (np.array([r["high"] - r["low"] for r in rows], float), tops.pop(),
            np.array([f'{r.get("tail") or "all"}|{r["cluster"]}' for r in rows]))

s = {m: {c: cell(m, c)[0].mean() / cell(m, c)[1] for c in CELLS} for m in MODELS}
for m in MODELS:
    print(f"{m:22s} " + "  ".join(f"{c} {s[m][c]:.3f}" for c in CELLS))

PRINTED_S = {"Gemini 3.5 Flash Lite": (0.037, 0.031, 0.016, 0.015),
             "GPT-4.1-mini": (0.038, 0.039, 0.031, 0.105),
             "Qwen3-235B": (0.076, 0.060, 0.069, 0.179)}
for m, want in PRINTED_S.items():
    for c, w in zip(CELLS, want):
        assert same(s[m][c], w, 3), (m, c, s[m][c], w)

# The text's derived figures.
g, q, gm = s["GPT-4.1-mini"], s["Qwen3-235B"], s["Gemini 3.5 Flash Lite"]
assert same(g["menagerie"] / g["jorgpt10"], 2.8, 1) and same(q["menagerie"] / q["jorgpt10"], 2.4, 1)
assert same(gm["menagerie"] / gm["jorgpt10"], 0.4, 1)
assert same(g["menagerie"] * 13, 1.4, 1) and same(q["menagerie"] * 13, 2.3, 1)
sd_men = np.std([r["human"] for r in cases["grader-label-stability/draw-menagerie"]])   # the draw's human SD
assert same(g["menagerie"] * 13 / sd_men, 0.56) and same(q["menagerie"] * 13 / sd_men, 0.94)
assert all(abs(m_["jorgpt5"] - m_["jorgpt10"]) <= 0.02 for m_ in (gm, g))
print(f"programs vs short answers: GPT-4.1-mini {g['menagerie'] / g['jorgpt10']:.1f}x, Qwen "
      f"{q['menagerie'] / q['jorgpt10']:.1f}x, Gemini {gm['menagerie'] / gm['jorgpt10']:.1f}x; "
      f"{g['menagerie'] * 13:.1f} and {q['menagerie'] * 13:.1f} points = {g['menagerie'] * 13 / sd_men:.2f} and "
      f"{q['menagerie'] * 13 / sd_men:.2f} SD of the human grades")
print("All 12 values of s and the derived figures match the paper.")

Gemini 3.5 Flash Lite  jorgpt10 0.037  jorgpt5 0.031  mohler 0.016  menagerie 0.015
GPT-4.1-mini           jorgpt10 0.038  jorgpt5 0.039  mohler 0.031  menagerie 0.105
Qwen3-235B             jorgpt10 0.076  jorgpt5 0.060  mohler 0.069  menagerie 0.179
programs vs short answers: GPT-4.1-mini 2.8x, Qwen 2.4x, Gemini 0.4x; 1.4 and 2.3 points = 0.56 and 0.94 SD of the human grades
All 12 values of s and the derived figures match the paper.


**Intervals and direction.** The study resamples clusters (JorGPT: students within tail; Mohler:
questions; Menagerie: submissions) 4,000 times, percentile 95%. Its draws are shared across cells and
contrasts on one stream, which this notebook does not reproduce, so the endpoints below come from the
same scheme on a different random stream and can differ from the printed ones in the last digit. The
direction claim (a high label raised the mark in all 13 model-by-corpus cells, GPT-5-mini's JorGPT
cell included) is checked on them.

In [8]:
def s_interval(d, top, cl, rng, reps=4000):
    strata = np.array([c.split("|")[0] for c in cl])
    return np.percentile([d[cluster_index(cl, strata, rng)].mean() / top for _ in range(reps)], [2.5, 97.5])

rng3 = np.random.default_rng(20261006)
lower = []
for m in MODELS:
    parts = []
    for c in CELLS:
        lo, up = s_interval(*cell(m, c), rng3)
        lower.append(lo)
        parts.append(f"{s[m][c]:.3f} [{lo:.3f}, {up:.3f}]")
    print(f"{m:22s} " + "   ".join(parts))
g5 = swap_rows("grader-vi/swap-gpt5mini")
lo5, up5 = s_interval(hl(g5), 10, np.array([f'{r["tail"]}|{r["learnerId"]}' for r in g5]), rng3)
lower.append(lo5)
print(f"GPT-5-mini, JorGPT 0–10: {hl(g5).mean() / 10:.3f} [{lo5:.3f}, {up5:.3f}]")
assert len(lower) == 13 and min(lower) > 0
print("All 13 cells positive: the high label raised the mark on every model, scale and item type tried.")

Gemini 3.5 Flash Lite  0.037 [0.030, 0.044]   0.031 [0.023, 0.038]   0.016 [0.006, 0.027]   0.015 [0.008, 0.023]
GPT-4.1-mini           0.038 [0.029, 0.045]   0.039 [0.030, 0.048]   0.031 [0.020, 0.042]   0.105 [0.094, 0.117]
Qwen3-235B             0.076 [0.070, 0.082]   0.060 [0.051, 0.068]   0.069 [0.059, 0.078]   0.179 [0.160, 0.197]
GPT-5-mini, JorGPT 0–10: 0.052 [0.043, 0.061]
All 13 cells positive: the high label raised the mark on every model, scale and item type tried.


## Claude Fable 5.1, run as an agent

Run `fable-agent/swap-fable-agent`: 100 answers, true and swapped labels only (no control arm), graded by
Claude Code subagents; one grading was not returned. Compared with Opus 5.5 via the API
(`breadth/swap-opus55`) on the same answers.

In [9]:
fab = [r for r in cases["fable-agent/swap-fable-agent"] if None not in (r["trueLabel"], r["swappedLabel"])]
opus = {r["id"]: r for r in cases["breadth/swap-opus55"] if None not in (r["trueLabel"], r["swappedLabel"])}
df, hf = hl(fab), np.array([r["tail"] == "high" for r in fab])
do = hl([opus[r["id"]] for r in fab])
print(f"Fable: {len(fab)} answers, L {df.mean():+.2f} (high {df[hf].mean():+.2f}, low {df[~hf].mean():+.2f}); "
      f"Opus 5.5 on the same answers {do.mean():+.2f}; difference {df.mean() - do.mean():+.2f}")
assert len(fab) == 99 and same(df.mean(), 0.31) and same(df[hf].mean(), 0.44) and same(df[~hf].mean(), 0.19)
assert same(do.mean(), 0.34) and same(df.mean() - do.mean(), -0.03)
print("The five Fable figures match the paper.")

Fable: 99 answers, L +0.31 (high +0.44, low +0.19); Opus 5.5 on the same answers +0.34; difference -0.03
The five Fable figures match the paper.
